# ML System Design for Senior Interviews

**Based on:** Chip Huyen's *Designing Machine Learning Systems* and CS 329S (Stanford)

ML System Design is the hardest part of senior/staff ML interviews. It tests whether you can
think end-to-end: from vague business requirements all the way to a production system.

**What interviewers assess:**
- Can you clarify ambiguous requirements?
- Do you understand data pipelines, not just models?
- Can you reason about scale, latency, and cost trade-offs?
- Do you know how systems fail in production?

---

**Notebook Structure:**
1. The 4-Step Framework
2. Case Study: Recommendation System
3. Case Study: Fraud Detection
4. Case Study: Search and Ranking
5. Scale and Infrastructure
6. 20 Interview Questions with Frameworks


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


---
## Section 1 — The ML System Design Framework

### The Golden Rule
An ML interview is not a quiz. It is a design conversation. Your job is to show
*how you think*, not to produce the perfect answer. Always start by asking
clarifying questions.


### The 4-Step Framework

```
+------------------+   +------------------+   +------------------+   +------------------+
|   1. PROBLEM     |-->|   2. DATA        |-->|   3. MODEL       |-->|   4. DEPLOYMENT  |
|                  |   |                  |   |                  |   |                  |
| - Business goal  |   | - Sources        |   | - Architecture   |   | - Serving        |
| - ML framing     |   | - Labeling       |   | - Features       |   | - Monitoring     |
| - Constraints    |   | - Pipeline       |   | - Training       |   | - Feedback loop  |
| - Metrics        |   | - Storage        |   | - Evaluation     |   | - A/B testing    |
+------------------+   +------------------+   +------------------+   +------------------+
```

Each step has a standard set of questions to ask.


### Step 1 — Problem Framing

**Questions to ask the interviewer:**

1. What is the *business* objective? (revenue, engagement, safety, cost?)
2. How will success be measured? (online metrics vs offline metrics)
3. What is the latency requirement? (real-time <100ms, near-real-time <1s, batch ok?)
4. What scale? (requests/sec, users, items in the catalog)
5. Are there fairness/legal/compliance constraints?
6. What ML systems exist today? (greenfield vs replacing something)
7. What data do we already have?

**ML Task Taxonomy:**
- **Classification**: spam, fraud, sentiment, content moderation
- **Regression**: price prediction, ETA, revenue forecast
- **Ranking/Retrieval**: search, recommendations, ads
- **Generation**: summarization, code completion, image synthesis
- **Anomaly Detection**: fraud, system health, data quality

**Business Metric vs ML Metric:**
| Business Metric | ML Metric |
|---|---|
| Click-through rate | AUC-ROC, log loss |
| Revenue | NDCG, Precision@K |
| User retention | MAP, Hit Rate |
| Fraud prevented ($) | Recall at fixed FPR |


### Step 2 — Data Strategy

**Key questions:**
1. How is data collected? (logs, user actions, sensors, crowdsourcing)
2. How is ground truth obtained? (explicit labels, implicit feedback, delayed labels)
3. How much data? (rule of thumb: 10x more than model parameters for simple models)
4. What are the labeling costs and timelines?
5. What are the privacy and compliance requirements (GDPR, HIPAA)?

**Data labeling strategies ranked by cost:**
```
Cheapest  |  Natural labels (click = positive, no-click = negative)
          |  Weak supervision (Snorkel, keyword heuristics)
          |  Semi-supervised (label propagation on small labeled set)
          |  Active learning (query most uncertain examples)
Expensive |  Human annotation (MTurk, in-house labelers)
```

**Data splits — the interview trap:**
- Never split randomly for time-series data — always split by time
- Train on past, validate on present, test on future
- Beware of leakage from future features into training


### Step 3 — Model Selection

**Feature engineering (often more impactful than model choice):**
- Numerical: normalization, log transforms, binning, interactions
- Categorical: one-hot, target encoding, embeddings
- Temporal: hour of day, day of week, recency, rolling aggregates
- Text: TF-IDF, embeddings (word2vec, BERT)
- Graph: degree, PageRank, community membership

**Model selection heuristic:**
```
Data size < 10k rows   -->  Logistic Regression, SVM, kNN
10k - 1M rows          -->  Gradient Boosting (XGBoost, LightGBM)
> 1M rows + structure  -->  Neural Networks, Deep Learning
Text / Images / Audio  -->  Transformer fine-tuning
Sparse interactions    -->  Matrix Factorization, Two-Tower NN
```

**Evaluation — always use multiple metrics:**
- Offline: AUC, NDCG, RMSE (depends on task)
- Online: A/B test with primary + guardrail metrics
- Slice analysis: does the model perform equally across subgroups?


### Step 4 — Deployment and Operations

**The deployment checklist:**
- [ ] Shadow mode: run new model in parallel, log but don't serve
- [ ] Canary deployment: route 1% traffic to new model
- [ ] Feature flag: ability to instantly roll back
- [ ] Monitoring: data drift, prediction drift, latency, error rate
- [ ] Retraining strategy: scheduled, triggered, or continuous

**Common failure modes in production:**
1. Training-serving skew (features computed differently at train vs serve time)
2. Distribution shift (world changes, model doesn't)
3. Feedback loops (model predictions affect future training data)
4. Underspecification (model picks up spurious correlations that break in prod)

**The feedback loop:**
```
User action --> Logging --> Feature pipeline --> Training --> Model update --> Serving
     ^                                                                            |
     +----------------------------------------------------------------------------+
```


### Capacity Estimation Template

This is often skipped in ML interviews but impresses interviewers at senior levels.

For a system with N users and R requests/user/day:


In [ ]:
# Capacity estimation calculator
def estimate_capacity(
    daily_active_users,
    requests_per_user_per_day,
    avg_request_size_bytes,
    avg_response_size_bytes,
    model_inference_time_ms,
    training_data_gb_per_month
):
    total_requests_per_day = daily_active_users * requests_per_user_per_day
    rps = total_requests_per_day / 86400  # requests per second
    peak_rps = rps * 5  # assume 5x peak factor

    ingress_gbps = (total_requests_per_day * avg_request_size_bytes) / (86400 * 1e9)
    egress_gbps = (total_requests_per_day * avg_response_size_bytes) / (86400 * 1e9)

    # Servers needed: each server handles ~1000 concurrent requests at 100ms latency
    concurrency = peak_rps * (model_inference_time_ms / 1000.0)
    servers_needed = max(1, int(concurrency / 100) + 1)

    print("=== Capacity Estimation ===")
    print(f"Daily requests:       {total_requests_per_day:,.0f}")
    print(f"Average RPS:          {rps:,.1f}")
    print(f"Peak RPS (5x):        {peak_rps:,.1f}")
    print(f"Ingress bandwidth:    {ingress_gbps*1000:.2f} MB/s")
    print(f"Egress bandwidth:     {egress_gbps*1000:.2f} MB/s")
    print(f"Concurrent requests:  {concurrency:,.1f}")
    print(f"Servers (rough est):  {servers_needed}")
    print(f"Training data/month:  {training_data_gb_per_month} GB")
    print()
    return {"rps": rps, "peak_rps": peak_rps, "servers": servers_needed}

# Example: Netflix recommendation system
print("--- Netflix-scale Recommendation System ---")
netflix = estimate_capacity(
    daily_active_users=200_000_000,
    requests_per_user_per_day=10,
    avg_request_size_bytes=500,
    avg_response_size_bytes=5000,
    model_inference_time_ms=50,
    training_data_gb_per_month=5000
)

# Example: Startup fraud detection
print("--- Startup Fraud Detection (10k merchants) ---")
startup = estimate_capacity(
    daily_active_users=10_000,
    requests_per_user_per_day=100,
    avg_request_size_bytes=2000,
    avg_response_size_bytes=200,
    model_inference_time_ms=20,
    training_data_gb_per_month=10
)


---
## Section 2 — Case Study: Recommendation System

### Interview Prompt
*"Design the recommendation system for Netflix/YouTube/Amazon."*

This is the most common ML system design question. Let's walk through it with real code.


### Problem Framing

**Business objective:** Maximize engagement (watch time / click-through)

**Key clarifications to ask:**
- Homepage recommendations or similar-items?
- Logged-in users only, or anonymous too?
- Latency: can we precompute, or does it need to be real-time?
- How many items in the catalog? (1k vs 100M changes architecture)
- Cold start: how do we handle new users and new items?

**ML framing options:**

| Approach | Pro | Con |
|---|---|---|
| Click prediction (classification) | Easy to train | Click != satisfaction |
| Rating prediction (regression) | Direct signal | Sparse ratings |
| Ranking (LTR) | Optimizes for order | Needs impression data |
| Retrieval + Ranking (two-stage) | Scalable | Complex pipeline |

**Two-stage architecture (used by YouTube, Pinterest, LinkedIn):**
```
All items (millions)
       |
  [Retrieval] -- fast, high recall, lower precision
       | ~1000 candidates
  [Ranking]  -- slower, high precision, personalized
       | Top 10-50
  [Re-ranking] -- apply diversity, freshness, business rules
       | Final list served
```


### Data Setup: MovieLens-style Dataset

We'll create a realistic small dataset resembling MovieLens 100k.


In [ ]:
# Create a synthetic MovieLens-style dataset
np.random.seed(42)

N_USERS = 500
N_MOVIES = 300
SPARSITY = 0.02  # 2% of all user-movie pairs have ratings

# Movie metadata
genres = ['Action', 'Comedy', 'Drama', 'Horror', 'Sci-Fi', 'Romance', 'Thriller']
movie_ids = list(range(N_MOVIES))
movie_genres = {mid: np.random.choice(genres) for mid in movie_ids}
movie_years = {mid: np.random.randint(1990, 2024) for mid in movie_ids}
movie_titles = {mid: f"Movie_{mid}" for mid in movie_ids}

# User-item interactions (sparse ratings 1-5)
n_ratings = int(N_USERS * N_MOVIES * SPARSITY)
user_ids_ratings = np.random.randint(0, N_USERS, n_ratings)
item_ids_ratings = np.random.randint(0, N_MOVIES, n_ratings)

# Add genre preferences per user to make it realistic
user_genre_pref = {uid: np.random.choice(genres) for uid in range(N_USERS)}

ratings_list = []
for u, i in zip(user_ids_ratings, item_ids_ratings):
    base_rating = 3.0
    if movie_genres[i] == user_genre_pref[u]:
        base_rating += 1.2
    rating = np.clip(base_rating + np.random.normal(0, 0.8), 1, 5)
    ratings_list.append({'user_id': u, 'movie_id': i, 'rating': round(rating * 2) / 2})

ratings_df = pd.DataFrame(ratings_list).drop_duplicates(subset=['user_id', 'movie_id'])
print(f"Total ratings: {len(ratings_df):,}")
print(f"Unique users:  {ratings_df['user_id'].nunique()}")
print(f"Unique movies: {ratings_df['movie_id'].nunique()}")
print(f"Sparsity:      {1 - len(ratings_df) / (N_USERS * N_MOVIES):.4f}")
print()
print(ratings_df.head(10).to_string(index=False))


### Collaborative Filtering — Matrix Factorization

**Idea:** Decompose the user-item rating matrix R into two low-rank matrices:
```
R (n_users x n_items) ~ U (n_users x k) x V^T (k x n_items)
```
Where k is the number of latent factors (e.g., 50). The latent factors capture
abstract concepts like "likes action movies" or "prefers 90s films".


In [ ]:
def build_rating_matrix(df, n_users, n_items):
    """Convert ratings dataframe to sparse user-item matrix."""
    matrix = np.zeros((n_users, n_items))
    for _, row in df.iterrows():
        matrix[int(row['user_id']), int(row['movie_id'])] = row['rating']
    return matrix

# Build the user-item matrix
R = build_rating_matrix(ratings_df, N_USERS, N_MOVIES)
print(f"Rating matrix shape: {R.shape}")
print(f"Non-zero entries: {np.count_nonzero(R):,}")
print(f"Mean rating (observed): {R[R > 0].mean():.3f}")

# Visualize rating distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ratings_df['rating'], bins=9, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Rating Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings per user distribution
ratings_per_user = ratings_df.groupby('user_id').size()
axes[1].hist(ratings_per_user, bins=30, color='coral', edgecolor='white', linewidth=0.5)
axes[1].set_title('Ratings per User', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(ratings_per_user.mean(), color='navy', linestyle='--',
                label=f'Mean: {ratings_per_user.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/recsys_data.png', dpi=100, bbox_inches='tight')
plt.show()
print("Rating matrix visualization saved.")


In [ ]:
class MatrixFactorizationSVD:
    """
    SVD-based collaborative filtering (similar to Simon Funk's SVD).
    Uses scipy's truncated SVD on the user-item rating matrix.
    """
    def __init__(self, n_factors=50):
        self.n_factors = n_factors
        self.U = None
        self.sigma = None
        self.Vt = None
        self.user_means = None
        self.global_mean = None

    def fit(self, R):
        """
        R: dense user-item matrix (rows=users, cols=items).
        Zero entries mean 'unobserved', not 'rated 0'.
        """
        self.global_mean = R[R > 0].mean()
        # Mean-center by user
        self.user_means = np.where(
            (R > 0).sum(axis=1, keepdims=True) > 0,
            R.sum(axis=1, keepdims=True) / np.maximum((R > 0).sum(axis=1, keepdims=True), 1),
            self.global_mean
        )
        R_centered = np.where(R > 0, R - self.user_means, 0)

        sparse_R = csr_matrix(R_centered)
        k = min(self.n_factors, min(sparse_R.shape) - 1)
        self.U, self.sigma, self.Vt = svds(sparse_R, k=k)
        # Sort by singular value (svds returns smallest first)
        idx = np.argsort(self.sigma)[::-1]
        self.U = self.U[:, idx]
        self.sigma = self.sigma[idx]
        self.Vt = self.Vt[idx, :]
        return self

    def predict_all(self):
        """Reconstruct full predicted rating matrix."""
        R_hat = self.U @ np.diag(self.sigma) @ self.Vt + self.user_means
        return np.clip(R_hat, 1, 5)

    def recommend(self, user_id, R_observed, top_n=10):
        """Return top-N item recommendations for a user."""
        R_hat = self.predict_all()
        user_preds = R_hat[user_id].copy()
        # Mask already-seen items
        seen_items = np.where(R_observed[user_id] > 0)[0]
        user_preds[seen_items] = -np.inf
        top_items = np.argsort(user_preds)[::-1][:top_n]
        scores = user_preds[top_items]
        return list(zip(top_items, scores))

# Train
mf = MatrixFactorizationSVD(n_factors=50)
mf.fit(R)
R_hat = mf.predict_all()

# Evaluate on observed ratings
observed_mask = R > 0
actual = R[observed_mask]
predicted = R_hat[observed_mask]
rmse = np.sqrt(np.mean((actual - predicted) ** 2))
mae = np.mean(np.abs(actual - predicted))

print(f"Matrix Factorization (SVD)")
print(f"  RMSE on training set: {rmse:.4f}")
print(f"  MAE  on training set: {mae:.4f}")
print()

# Example recommendations for user 0
recs = mf.recommend(user_id=0, R_observed=R, top_n=5)
print(f"Top-5 recommendations for User 0:")
print(f"  (User 0 prefers genre: {user_genre_pref[0]})")
for movie_id, score in recs:
    print(f"  Movie {movie_id:3d} | Genre: {movie_genres[movie_id]:10s} | "
          f"Year: {movie_years[movie_id]} | Predicted Rating: {score:.2f}")


### Content-Based Filtering

Uses item features (genre, year, description) to find similar items.
No user interaction data needed — solves the cold-start problem for new items.


In [ ]:
def build_item_features(movie_ids, movie_genres, movie_years):
    """Build a feature matrix for items."""
    genre_list = sorted(set(movie_genres.values()))
    year_min, year_max = 1990, 2024

    rows = []
    for mid in movie_ids:
        genre_vec = [1.0 if movie_genres[mid] == g else 0.0 for g in genre_list]
        year_norm = (movie_years[mid] - year_min) / (year_max - year_min)
        rows.append(genre_vec + [year_norm])

    return np.array(rows), genre_list + ['year_norm']

item_features, feature_names = build_item_features(movie_ids, movie_genres, movie_years)
print(f"Item feature matrix shape: {item_features.shape}")
print(f"Features: {feature_names}")

# Compute item-item similarity
item_sim_matrix = cosine_similarity(item_features)
print(f"Item similarity matrix shape: {item_sim_matrix.shape}")

def content_based_recommend(user_id, R_observed, item_sim_matrix, top_n=10):
    """
    For a user, find items similar to what they liked.
    Weighted by rating given to liked items.
    """
    user_ratings = R_observed[user_id]
    liked_items = np.where(user_ratings >= 4.0)[0]

    if len(liked_items) == 0:
        # Fall back to global popular items
        popularity = (R_observed > 0).sum(axis=0)
        return list(np.argsort(popularity)[::-1][:top_n])

    # Score each item by weighted similarity to liked items
    scores = np.zeros(item_sim_matrix.shape[0])
    for item in liked_items:
        scores += user_ratings[item] * item_sim_matrix[item]

    # Mask seen items
    seen = np.where(user_ratings > 0)[0]
    scores[seen] = -np.inf

    top_items = np.argsort(scores)[::-1][:top_n]
    return [(i, scores[i]) for i in top_items]

cb_recs = content_based_recommend(0, R, item_sim_matrix, top_n=5)
print(f"\nContent-Based Top-5 for User 0:")
print(f"  (User 0 prefers genre: {user_genre_pref[0]})")
for movie_id, score in cb_recs:
    print(f"  Movie {movie_id:3d} | Genre: {movie_genres[movie_id]:10s} | Score: {score:.3f}")


### Hybrid Approach

In production, a hybrid approach almost always beats either pure CF or pure CB.


In [ ]:
def hybrid_recommend(user_id, R_observed, R_hat_mf, item_sim_matrix,
                     alpha=0.7, top_n=10):
    """
    Hybrid: alpha * MF_score + (1-alpha) * CB_score
    alpha = 0.7 gives more weight to collaborative filtering.
    For cold-start users (few ratings), reduce alpha.
    """
    n_user_ratings = (R_observed[user_id] > 0).sum()
    # Dynamic alpha: trust CF more when we have more data
    if n_user_ratings < 5:
        alpha = 0.2  # mostly content-based for cold start
    elif n_user_ratings < 20:
        alpha = 0.5

    # MF scores
    mf_scores = R_hat_mf[user_id].copy()

    # CB scores
    user_ratings = R_observed[user_id]
    liked_items = np.where(user_ratings >= 4.0)[0]
    cb_scores = np.zeros(item_sim_matrix.shape[0])
    if len(liked_items) > 0:
        for item in liked_items:
            cb_scores += user_ratings[item] * item_sim_matrix[item]

    # Normalize both to [0, 1]
    def minmax(arr):
        rng = arr.max() - arr.min()
        if rng == 0:
            return np.zeros_like(arr)
        return (arr - arr.min()) / rng

    mf_norm = minmax(mf_scores)
    cb_norm = minmax(cb_scores)

    hybrid_scores = alpha * mf_norm + (1 - alpha) * cb_norm

    # Mask seen items
    seen = np.where(R_observed[user_id] > 0)[0]
    hybrid_scores[seen] = -np.inf

    top_items = np.argsort(hybrid_scores)[::-1][:top_n]
    return [(i, hybrid_scores[i]) for i in top_items]

# Compare approaches for user 0
print(f"=== Comparison for User 0 (prefers {user_genre_pref[0]}) ===")
print()

mf_recs = mf.recommend(0, R, top_n=5)
cb_recs_5 = content_based_recommend(0, R, item_sim_matrix, top_n=5)
hyb_recs = hybrid_recommend(0, R, R_hat, item_sim_matrix, top_n=5)

print(f"{'Rank':<5} {'Matrix Factorization':<25} {'Content-Based':<25} {'Hybrid':<25}")
print("-" * 80)
for rank in range(5):
    mf_g = movie_genres[mf_recs[rank][0]] if rank < len(mf_recs) else ""
    cb_g = movie_genres[cb_recs_5[rank][0]] if rank < len(cb_recs_5) else ""
    hy_g = movie_genres[hyb_recs[rank][0]] if rank < len(hyb_recs) else ""
    print(f"{rank+1:<5} {mf_g:<25} {cb_g:<25} {hy_g:<25}")


### Cold Start Problem

**The cold start problem:** What do we do for new users or new items with no interaction history?

| Scenario | Strategy |
|---|---|
| New user | Ask onboarding questions, use demographic features, show popular/trending |
| New item | Use content features (CB filtering), propagate similar item embeddings |
| New user + New item | Popularity-based, exploration strategies |

**Exploration vs Exploitation:**
- Epsilon-greedy: 90% exploit (best known), 10% explore (random)
- Upper Confidence Bound (UCB): prefer items with high uncertainty
- Thompson Sampling: Bayesian approach, naturally balances E&E


### Ranking Metrics: NDCG, MAP, Hit Rate


In [ ]:
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at k."""
    relevances = np.array(relevances[:k], dtype=float)
    if len(relevances) == 0:
        return 0.0
    gains = (2 ** relevances - 1) / np.log2(np.arange(2, len(relevances) + 2))
    return gains.sum()

def ndcg_at_k(recommended, relevant_items, k=10):
    """
    Normalized DCG.
    recommended: ordered list of item ids
    relevant_items: dict of {item_id: relevance_score} (or set for binary)
    """
    if isinstance(relevant_items, set):
        relevant_items = {i: 1.0 for i in relevant_items}

    gains = [relevant_items.get(item, 0.0) for item in recommended[:k]]
    ideal = sorted(relevant_items.values(), reverse=True)

    actual_dcg = dcg_at_k(gains, k)
    ideal_dcg = dcg_at_k(ideal, k)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

def average_precision_at_k(recommended, relevant_items, k=10):
    """Average Precision at k."""
    if isinstance(relevant_items, (set, dict)):
        relevant_set = set(relevant_items)
    hits = 0
    precision_sum = 0.0
    for idx, item in enumerate(recommended[:k]):
        if item in relevant_set:
            hits += 1
            precision_sum += hits / (idx + 1)
    return precision_sum / min(len(relevant_set), k) if relevant_set else 0.0

def hit_rate_at_k(recommended, relevant_items, k=10):
    """Hit Rate: was at least one relevant item in top-k?"""
    relevant_set = set(relevant_items)
    return 1.0 if any(item in relevant_set for item in recommended[:k]) else 0.0

# Evaluate on held-out test set
# For each user, hide their top-rated movies and see if the model recovers them
test_users = list(range(50))  # evaluate on first 50 users
ndcg_scores = []
map_scores = []
hr_scores = []

for uid in test_users:
    user_ratings = ratings_df[ratings_df['user_id'] == uid]
    if len(user_ratings) < 5:
        continue

    # Hold out top 20% as test set
    user_ratings_sorted = user_ratings.sort_values('rating', ascending=False)
    n_test = max(1, int(len(user_ratings) * 0.2))
    test_items = set(user_ratings_sorted.head(n_test)['movie_id'].astype(int))
    test_relevances = {int(row['movie_id']): float(row['rating'])
                      for _, row in user_ratings_sorted.head(n_test).iterrows()}

    # Build training matrix (without test items)
    R_train = R.copy()
    for item in test_items:
        R_train[uid, item] = 0

    recs = [item for item, _ in mf.recommend(uid, R_train, top_n=20)]
    ndcg_scores.append(ndcg_at_k(recs, test_relevances, k=10))
    map_scores.append(average_precision_at_k(recs, test_items, k=10))
    hr_scores.append(hit_rate_at_k(recs, test_items, k=10))

print("=== Recommendation System Evaluation ===")
print(f"Evaluated on {len(ndcg_scores)} users")
print(f"NDCG@10:     {np.mean(ndcg_scores):.4f} +/- {np.std(ndcg_scores):.4f}")
print(f"MAP@10:      {np.mean(map_scores):.4f} +/- {np.std(map_scores):.4f}")
print(f"HitRate@10:  {np.mean(hr_scores):.4f} +/- {np.std(hr_scores):.4f}")
print()
print("Interpretation:")
print(f"  {np.mean(hr_scores)*100:.1f}% of the time, at least one held-out item")
print(f"  was recovered in the top-10 recommendations.")


---
## Section 3 — Case Study: Fraud Detection System

### Interview Prompt
*"Design a real-time fraud detection system for a payments company."*

Fraud detection is a canonical example of **high-stakes, imbalanced classification**
with extreme operational constraints.


### Problem Framing

**Key clarifications to ask:**
1. Real-time (per-transaction, <100ms) or batch (daily review)?
2. What type of fraud? (account takeover, card fraud, merchant fraud, promo abuse)
3. What is the cost asymmetry? (false positive = blocked legit user, false negative = fraud loss)
4. What actions does the system take? (block, flag for review, add friction/2FA)
5. What signals do we have? (transaction data, device, location, behavior history)
6. Regulatory constraints? (PCI-DSS, fair lending laws)

**Decision tiers:**
```
Score >= 0.9  -->  Auto-block (high confidence fraud)
Score 0.5-0.9 -->  Manual review queue
Score 0.2-0.5 -->  Add friction (2FA, step-up auth)
Score < 0.2   -->  Auto-approve
```

**Why this is hard:**
- Class imbalance: 0.1-2% fraud rate
- Adversarial: fraudsters adapt to the model
- Delayed labels: chargebacks take 30-90 days to arrive
- Low latency: must decide in <100ms
- High cost asymmetry: wrong decisions have different costs


### Feature Engineering for Fraud

Features are more important than the model for fraud detection.


In [ ]:
# Simulate a realistic fraud dataset
np.random.seed(42)

n_transactions = 50000
fraud_rate = 0.015  # 1.5% fraud rate
n_fraud = int(n_transactions * fraud_rate)
n_legit = n_transactions - n_fraud

print(f"Generating {n_transactions:,} transactions ({fraud_rate*100}% fraud rate)")
print(f"  Legitimate: {n_legit:,}")
print(f"  Fraudulent: {n_fraud:,}")
print()

def generate_fraud_features(n_legit, n_fraud):
    """
    Generate synthetic transaction features.
    Fraudulent transactions have distinct statistical signatures.
    """
    # Legitimate transactions
    legit = {
        'amount': np.random.lognormal(4.5, 1.0, n_legit),   # Typical purchase amounts
        'hour_of_day': np.random.choice(range(24), n_legit,
                        p=np.array([1,1,1,1,1,1,2,4,6,7,7,7,7,6,6,7,7,6,5,4,3,2,2,1]) / 83.0),
        'dist_from_home_km': np.abs(np.random.normal(10, 15, n_legit)),
        'transactions_last_1h': np.random.poisson(0.5, n_legit),
        'transactions_last_24h': np.random.poisson(5, n_legit),
        'is_new_merchant': np.random.binomial(1, 0.1, n_legit),
        'failed_attempts_last_24h': np.random.poisson(0.1, n_legit),
        'avg_amount_30d': np.random.lognormal(4.3, 0.8, n_legit),
        'days_since_last_transaction': np.random.exponential(1.5, n_legit),
        'same_country': np.random.binomial(1, 0.95, n_legit),
        'label': np.zeros(n_legit)
    }

    # Fraudulent transactions (different patterns)
    fraud = {
        'amount': np.random.lognormal(5.5, 1.5, n_fraud),   # Higher, more variable
        'hour_of_day': np.random.choice(range(24), n_fraud,
                        p=np.array([4,5,5,5,5,4,2,2,2,2,2,2,2,2,2,2,2,2,2,3,4,5,5,5]) / 74.0),
        'dist_from_home_km': np.abs(np.random.normal(200, 300, n_fraud)),  # Far from home
        'transactions_last_1h': np.random.poisson(3, n_fraud),  # Velocity spike
        'transactions_last_24h': np.random.poisson(12, n_fraud),
        'is_new_merchant': np.random.binomial(1, 0.6, n_fraud),  # New merchants
        'failed_attempts_last_24h': np.random.poisson(2, n_fraud),
        'avg_amount_30d': np.random.lognormal(4.0, 0.8, n_fraud),
        'days_since_last_transaction': np.random.exponential(0.5, n_fraud),
        'same_country': np.random.binomial(1, 0.4, n_fraud),  # Cross-border
        'label': np.ones(n_fraud)
    }

    df_legit = pd.DataFrame(legit)
    df_fraud = pd.DataFrame(fraud)
    df = pd.concat([df_legit, df_fraud], ignore_index=True).sample(frac=1, random_state=42)
    df = df.reset_index(drop=True)
    return df

fraud_df = generate_fraud_features(n_legit, n_fraud)

# Engineer velocity and ratio features
fraud_df['amount_to_avg_ratio'] = fraud_df['amount'] / (fraud_df['avg_amount_30d'] + 1e-6)
fraud_df['velocity_spike'] = fraud_df['transactions_last_1h'] / (
    fraud_df['transactions_last_24h'] / 24 + 0.01)
fraud_df['log_amount'] = np.log1p(fraud_df['amount'])
fraud_df['log_dist'] = np.log1p(fraud_df['dist_from_home_km'])

print("Feature summary:")
feature_cols = [c for c in fraud_df.columns if c != 'label']
print(fraud_df[feature_cols].describe().round(2).to_string())
print(f"\nFraud label distribution:")
print(fraud_df['label'].value_counts())


### Handling Class Imbalance

Three main strategies for dealing with imbalanced classes:
1. **Resampling**: oversample minority or undersample majority
2. **Class weights**: penalize misclassification of minority more
3. **Threshold tuning**: change decision threshold based on business cost



In [ ]:
from sklearn.utils import resample

feature_cols = ['amount', 'hour_of_day', 'dist_from_home_km', 'transactions_last_1h',
                'transactions_last_24h', 'is_new_merchant', 'failed_attempts_last_24h',
                'avg_amount_30d', 'days_since_last_transaction', 'same_country',
                'amount_to_avg_ratio', 'velocity_spike', 'log_amount', 'log_dist']

X = fraud_df[feature_cols].values
y = fraud_df['label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Strategy 1: No resampling (baseline)
clf_baseline = LogisticRegression(max_iter=1000, random_state=42)
clf_baseline.fit(X_train_scaled, y_train)

# Strategy 2: Class weights (built-in)
clf_weighted = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_weighted.fit(X_train_scaled, y_train)

# Strategy 3: Oversample minority class (SMOTE-like, using resample)
X_train_df = pd.DataFrame(X_train_scaled)
X_train_df['label'] = y_train
majority = X_train_df[X_train_df['label'] == 0]
minority = X_train_df[X_train_df['label'] == 1]
minority_upsampled = resample(minority, n_samples=len(majority), random_state=42)
balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42)
X_train_bal = balanced.drop('label', axis=1).values
y_train_bal = balanced['label'].values

clf_resampled = LogisticRegression(max_iter=1000, random_state=42)
clf_resampled.fit(X_train_bal, y_train_bal)

# Evaluate all three
def eval_model(name, clf, X_test, y_test):
    probs = clf.predict_proba(X_test)[:, 1]
    preds = clf.predict(X_test)
    auc = roc_auc_score(y_test, probs)
    ap = average_precision_score(y_test, probs)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"  {name:<35} AUC={auc:.4f}  AP={ap:.4f}  "
          f"Prec={precision:.3f}  Rec={recall:.3f}  "
          f"TP={tp}  FP={fp}  FN={fn}")
    return probs

print("=== Imbalance Strategy Comparison ===")
probs_base = eval_model("Baseline (no resampling)", clf_baseline, X_test_scaled, y_test)
probs_wt = eval_model("Class weights (balanced)", clf_weighted, X_test_scaled, y_test)
probs_rs = eval_model("Oversampling (minority)", clf_resampled, X_test_scaled, y_test)


### Threshold Optimization for Business Cost

In fraud detection, false negatives (missed fraud) and false positives (blocked customers)
have very different costs. We should choose the threshold based on business economics.


In [ ]:
# Business cost model
COST_FALSE_NEGATIVE = 150   # avg fraud loss if not caught ($)
COST_FALSE_POSITIVE = 15    # customer friction, support cost ($)
REVENUE_TRUE_POSITIVE = 5   # value of catching fraud (investigation savings, etc)

def compute_business_cost(y_true, y_prob, threshold,
                           cost_fn=150, cost_fp=15, rev_tp=5):
    """Compute expected business cost at a given threshold."""
    preds = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    total_cost = fn * cost_fn + fp * cost_fp - tp * rev_tp
    return total_cost, tp, fp, fn, tn

thresholds = np.linspace(0.01, 0.99, 100)
costs_wt = []
tpr_list, fpr_list = [], []

for t in thresholds:
    cost, tp, fp, fn, tn = compute_business_cost(y_test, probs_wt, t)
    costs_wt.append(cost)
    n_pos = tp + fn
    n_neg = tn + fp
    tpr_list.append(tp / n_pos if n_pos > 0 else 0)
    fpr_list.append(fp / n_neg if n_neg > 0 else 0)

optimal_idx = np.argmin(costs_wt)
optimal_threshold = thresholds[optimal_idx]
min_cost = costs_wt[optimal_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(thresholds, costs_wt, color='crimson', linewidth=2)
axes[0].axvline(optimal_threshold, color='navy', linestyle='--',
                label=f'Optimal threshold = {optimal_threshold:.2f}')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Expected Business Cost ($)')
axes[0].set_title('Business Cost vs Decision Threshold', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC curve comparison
fpr_base, tpr_base, _ = roc_curve(y_test, probs_base)
fpr_wt2, tpr_wt2, _ = roc_curve(y_test, probs_wt)
fpr_rs2, tpr_rs2, _ = roc_curve(y_test, probs_rs)

axes[1].plot(fpr_base, tpr_base, label=f'Baseline (AUC={roc_auc_score(y_test, probs_base):.3f})')
axes[1].plot(fpr_wt2, tpr_wt2, label=f'Weighted (AUC={roc_auc_score(y_test, probs_wt):.3f})')
axes[1].plot(fpr_rs2, tpr_rs2, label=f'Oversampled (AUC={roc_auc_score(y_test, probs_rs):.3f})')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/fraud_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Optimal decision threshold: {optimal_threshold:.3f}")
print(f"Expected cost at optimal: ${min_cost:,.0f}")
print(f"Cost at default threshold (0.5): ${compute_business_cost(y_test, probs_wt, 0.5)[0]:,.0f}")


### Feature Importance for Fraud Detection


In [ ]:
# Train a gradient boosting model and examine feature importance
clf_gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
clf_gb.fit(X_train_scaled, y_train)

probs_gb = clf_gb.predict_proba(X_test_scaled)[:, 1]
print(f"GBM AUC: {roc_auc_score(y_test, probs_gb):.4f}")
print(f"GBM AP:  {average_precision_score(y_test, probs_gb):.4f}")

# Feature importances
importances = clf_gb.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue' if imp < 0.1 else 'coral' for imp in feat_imp['importance']]
ax.barh(feat_imp['feature'], feat_imp['importance'], color=colors)
ax.set_title('Feature Importances (Gradient Boosting)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance')
ax.axvline(0.05, color='gray', linestyle='--', alpha=0.5, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/fraud_features.png', dpi=100, bbox_inches='tight')
plt.show()


### Model Monitoring for Fraud

Fraud patterns shift faster than almost any other ML domain.
Fraudsters actively adapt to defeat detection models.

**Key monitoring signals:**

| Signal | What it detects | Alert threshold |
|---|---|---|
| Fraud rate (daily) | Sudden spike in fraud | >2x rolling average |
| Feature drift | Input distribution change | PSI > 0.25 |
| Score distribution | Model score histogram shift | KS-test p < 0.01 |
| False negative rate | Model degradation | >1.5x baseline |
| Chargeback rate | Lagged ground truth | >0.5% of volume |

**Population Stability Index (PSI):**


In [ ]:
def compute_psi(expected, actual, n_bins=10):
    """
    Population Stability Index.
    PSI < 0.1:  No significant shift
    PSI 0.1-0.25: Moderate shift, investigate
    PSI > 0.25:  Major shift, retrain
    """
    # Create bins from expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    # Avoid log(0) and division by zero
    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

# Simulate distribution drift over time
np.random.seed(123)
train_scores = probs_gb[:5000]  # reference window

# Week 1: stable
week1_scores = train_scores + np.random.normal(0, 0.02, len(train_scores))

# Week 4: moderate drift (fraudsters adapting)
week4_scores = train_scores * 0.85 + np.random.normal(0.05, 0.05, len(train_scores))

# Week 8: major drift
week8_scores = train_scores * 0.6 + np.random.normal(0.15, 0.1, len(train_scores))

psi_w1 = compute_psi(train_scores, np.clip(week1_scores, 0, 1))
psi_w4 = compute_psi(train_scores, np.clip(week4_scores, 0, 1))
psi_w8 = compute_psi(train_scores, np.clip(week8_scores, 0, 1))

print("=== Score Distribution Drift (PSI) ===")
print(f"Week 1 PSI: {psi_w1:.4f}  --> {'STABLE' if psi_w1 < 0.1 else 'WARNING'}")
print(f"Week 4 PSI: {psi_w4:.4f}  --> {'STABLE' if psi_w4 < 0.1 else 'MODERATE DRIFT' if psi_w4 < 0.25 else 'MAJOR DRIFT'}")
print(f"Week 8 PSI: {psi_w8:.4f}  --> {'STABLE' if psi_w8 < 0.1 else 'MODERATE DRIFT' if psi_w8 < 0.25 else 'MAJOR DRIFT - RETRAIN NOW'}")


---
## Section 4 — Case Study: Search and Ranking

### Interview Prompt
*"Design the search system for an e-commerce site like eBay."*

Search is a retrieval + ranking pipeline. The key insight is that **retrieval**
and **ranking** have different computational budgets.


### Architecture Overview

```
User query: "red running shoes size 10"
                  |
        [Query Understanding]
        - Spell correction
        - Synonym expansion ("sneakers")
        - Intent classification (product search)
        - Entity extraction (color=red, type=running shoes, size=10)
                  |
        [Retrieval] (must be fast, ANN search)
        - Inverted index (BM25/TF-IDF)
        - Dense retrieval (two-tower embedding)
        - ~10k candidates
                  |
        [Ranking] (can be slower, ~10k -> 100)
        - LambdaMART / XGBoost with rich features
        - User personalization
        - ~100 results
                  |
        [Re-ranking / Blending]
        - Business rules (boost sponsored, suppress out-of-stock)
        - Diversity injection
        - Final 20 results
```


### TF-IDF Based Document Retrieval


In [ ]:
# Create a synthetic product catalog
np.random.seed(42)

product_catalog = [
    {"id": i, "title": title, "category": cat, "price": price, "rating": rating}
    for i, (title, cat, price, rating) in enumerate([
        ("Nike Air Max Running Shoes Men Size 10", "Footwear", 120.0, 4.5),
        ("Adidas Ultraboost Running Sneakers Red", "Footwear", 180.0, 4.7),
        ("Red Running Shoes Women Athletic", "Footwear", 75.0, 4.2),
        ("Under Armour Training Shoes Red Size 10", "Footwear", 95.0, 4.3),
        ("New Balance 990 Running Shoe", "Footwear", 175.0, 4.8),
        ("Blue Casual Sneakers Size 9", "Footwear", 60.0, 4.0),
        ("Running Shorts Men Athletic", "Apparel", 35.0, 4.1),
        ("Yoga Mat Non-slip Premium", "Sports", 45.0, 4.6),
        ("Protein Powder Vanilla 5lb", "Nutrition", 55.0, 4.4),
        ("Bluetooth Headphones Running Waterproof", "Electronics", 89.0, 4.3),
        ("Red Basketball Shoes High Top", "Footwear", 110.0, 4.2),
        ("Trail Running Shoes Waterproof Men", "Footwear", 140.0, 4.5),
        ("Compression Running Socks Red", "Apparel", 18.0, 4.3),
        ("GPS Running Watch Garmin", "Electronics", 299.0, 4.7),
        ("Foam Roller Recovery Tool", "Sports", 28.0, 4.4),
        ("Women's Jogging Shoes Size 8 Red", "Footwear", 85.0, 4.1),
        ("Cross Training Shoes Men Wide Fit", "Footwear", 100.0, 4.0),
        ("Sports Bra Running Women Red", "Apparel", 40.0, 4.5),
        ("Energy Gel Running Pack 24", "Nutrition", 32.0, 4.6),
        ("Insole Running Shoe Orthotic", "Footwear", 22.0, 4.2),
    ])
]

catalog_df = pd.DataFrame(product_catalog)
print(f"Product catalog: {len(catalog_df)} items")
print(catalog_df.head(5).to_string(index=False))


In [ ]:
# Build TF-IDF retrieval index
class TFIDFSearchEngine:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),   # unigrams and bigrams
            max_features=5000,
            stop_words='english',
            sublinear_tf=True     # log(1 + tf) instead of raw tf
        )
        self.doc_matrix = None
        self.docs = None

    def index(self, documents):
        """Build the inverted index."""
        self.docs = documents
        corpus = [doc['title'] + ' ' + doc['category'] for doc in documents]
        self.doc_matrix = self.vectorizer.fit_transform(corpus)
        print(f"Indexed {len(documents)} documents")
        print(f"Vocabulary size: {len(self.vectorizer.vocabulary_):,}")
        return self

    def search(self, query, top_k=5, return_scores=True):
        """BM25-like TF-IDF retrieval."""
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.doc_matrix)[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = [(self.docs[i], float(scores[i])) for i in top_indices if scores[i] > 0]
        return results

    def explain_match(self, query, doc_id):
        """Show which query terms matched and contributed most."""
        query_vec = self.vectorizer.transform([query])
        doc_vec = self.doc_matrix[doc_id]
        # Get feature names
        feature_names = self.vectorizer.get_feature_names_out()
        query_nz = query_vec.nonzero()[1]
        doc_nz = set(doc_vec.nonzero()[1])
        matches = []
        for idx in query_nz:
            if idx in doc_nz:
                score = query_vec[0, idx] * doc_vec[0, idx]
                matches.append((feature_names[idx], float(score)))
        return sorted(matches, key=lambda x: x[1], reverse=True)

engine = TFIDFSearchEngine()
engine.index(product_catalog)

# Run searches
test_queries = [
    "red running shoes size 10",
    "waterproof trail running",
    "nutrition for runners",
]

print()
for query in test_queries:
    print(f"Query: '{query}'")
    results = engine.search(query, top_k=3)
    for doc, score in results:
        print(f"  [{score:.3f}] {doc['title']:<50} ${doc['price']:.0f}")
    print()


### Learning to Rank

Beyond retrieval, ranking models personalize results using user signals.

**LambdaMART (used by Bing, Yahoo):**
- Gradient boosted trees trained with a ranking loss (LambdaRank)
- Optimizes NDCG directly
- Handles thousands of features (user, query, document, interaction)

**Key ranking features:**


In [ ]:
# Simulate a Learning-to-Rank dataset
# Each row is a (query, document) pair with features and relevance label

def generate_ltr_dataset(n_queries=200, n_docs_per_query=20):
    """
    Simulate LTR data.
    Relevance: 0=not relevant, 1=somewhat, 2=relevant, 3=highly relevant
    """
    rows = []
    for qid in range(n_queries):
        # Query-level signals
        query_len = np.random.randint(1, 8)
        query_has_brand = np.random.binomial(1, 0.3)

        for doc_rank in range(n_docs_per_query):
            # Document features
            doc_age_days = np.random.exponential(180)
            doc_price = np.random.lognormal(4, 1)
            doc_rating = np.random.uniform(1, 5)
            doc_n_reviews = int(np.random.exponential(100))

            # Query-document matching features
            title_match_score = np.random.beta(2, 5)  # TF-IDF score
            exact_title_match = np.random.binomial(1, 0.1)
            category_match = np.random.binomial(1, 0.6)

            # User-document interaction features
            ctr_28d = np.random.beta(2, 20)  # click-through rate
            add_to_cart_rate = ctr_28d * np.random.uniform(0.05, 0.3)

            # Compute noisy relevance (ground truth)
            relevance_signal = (
                2.0 * title_match_score +
                1.5 * exact_title_match +
                0.8 * category_match +
                0.5 * (doc_rating / 5.0) +
                0.3 * np.log1p(doc_n_reviews) / 7.0 +
                1.0 * ctr_28d * 10 +
                np.random.normal(0, 0.5)
            )
            relevance = int(np.clip(relevance_signal, 0, 3))

            rows.append({
                'qid': qid,
                'relevance': relevance,
                'query_len': query_len,
                'query_has_brand': query_has_brand,
                'doc_age_days': doc_age_days,
                'doc_price': doc_price,
                'doc_rating': doc_rating,
                'doc_n_reviews': doc_n_reviews,
                'title_match_score': title_match_score,
                'exact_title_match': exact_title_match,
                'category_match': category_match,
                'ctr_28d': ctr_28d,
                'add_to_cart_rate': add_to_cart_rate,
            })
    return pd.DataFrame(rows)

ltr_df = generate_ltr_dataset()
print(f"LTR dataset: {len(ltr_df):,} query-document pairs")
print(f"Queries: {ltr_df['qid'].nunique()}")
print(f"Relevance distribution:")
print(ltr_df['relevance'].value_counts().sort_index())


In [ ]:
# Pointwise ranking: treat as regression/classification
# (LambdaMART is pairwise/listwise, but pointwise is good to understand first)

ltr_features = ['query_len', 'query_has_brand', 'doc_age_days', 'doc_price',
                'doc_rating', 'doc_n_reviews', 'title_match_score',
                'exact_title_match', 'category_match', 'ctr_28d', 'add_to_cart_rate']

X_ltr = ltr_df[ltr_features].values
y_ltr = ltr_df['relevance'].values

# Train/test split by query ID (important! never split within a query)
all_qids = ltr_df['qid'].unique()
np.random.shuffle(all_qids)
train_qids = set(all_qids[:160])
test_qids = set(all_qids[160:])

train_mask = ltr_df['qid'].isin(train_qids)
X_ltr_train = X_ltr[train_mask]
y_ltr_train = y_ltr[train_mask]
X_ltr_test = X_ltr[~train_mask]
y_ltr_test = y_ltr[~train_mask]
qids_test = ltr_df[~train_mask]['qid'].values

# Train pointwise ranker (GBM)
ltr_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
ltr_model.fit(X_ltr_train, y_ltr_train)
ltr_scores = ltr_model.predict_proba(X_ltr_test)

# Combine scores into a single ranking score (weighted sum of class probs)
rank_scores = ltr_scores @ np.array([0, 0.33, 0.67, 1.0])

# Evaluate with NDCG
test_df = ltr_df[~train_mask].copy()
test_df['pred_score'] = rank_scores

ndcg_per_query = []
for qid in test_qids[:40]:
    q_df = test_df[test_df['qid'] == qid].sort_values('pred_score', ascending=False)
    if len(q_df) == 0:
        continue
    relevances = dict(zip(range(len(q_df)), q_df['relevance'].values))
    recommended = list(range(len(q_df)))
    ndcg_per_query.append(ndcg_at_k(recommended, relevances, k=10))

print(f"Pointwise LTR Model Evaluation")
print(f"  NDCG@10 (mean): {np.mean(ndcg_per_query):.4f}")
print(f"  NDCG@10 (std):  {np.std(ndcg_per_query):.4f}")
print()
print("Feature importances:")
feat_imp_ltr = pd.DataFrame({
    'feature': ltr_features,
    'importance': ltr_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp_ltr.to_string(index=False))


### Two-Tower Model Architecture

The two-tower model is the dominant architecture for large-scale retrieval
(used by YouTube, Google, Pinterest, Airbnb).

```
User Tower                    Item Tower
-----------                   -----------
user_id embedding             item_id embedding
user_features (age, loc)      item_features (title, category)
behavioral features           item statistics
       |                              |
   [MLP layers]               [MLP layers]
       |                              |
   user_embedding (d)          item_embedding (d)
       \_____________  dot  __________/
                   relevance score
```

**Why two towers?**
- Item tower can be pre-computed offline (all items)
- At query time, only run user tower + ANN search
- Scales to 100M+ items with millisecond latency

**Training signal:**
- Positive: user clicked / purchased
- Negative: random items (in-batch negatives) + hard negatives

**Offline ANN (Approximate Nearest Neighbor) libraries:**
- FAISS (Facebook/Meta) — GPU-accelerated
- ScaNN (Google) — best accuracy/speed trade-off
- HNSW (Hnswlib) — great for CPU


### Online vs Offline Evaluation

| Evaluation Type | Metric | Pros | Cons |
|---|---|---|---|
| Offline | NDCG, MAP, MRR | Fast, cheap, reproducible | May not reflect real behavior |
| Online (A/B test) | CTR, session length, revenue | Ground truth | Slow, costly, risky |
| Interleaving | Prefer rate | Faster than A/B | Complex to implement |
| Counterfactual | IPS-corrected metrics | No traffic risk | Requires logging |

**Guardrail metrics in A/B tests:**
- Revenue per user (must not decrease)
- Query abandonment rate (must not increase)
- Diversity (should not collapse to popular items)


---
## Section 5 — Scale and Infrastructure

### Batch vs Streaming Pipelines

One of the most important architectural decisions in ML systems.


### When to Use Batch vs Streaming

| Dimension | Batch | Streaming |
|---|---|---|
| Latency | Hours to days | Milliseconds to seconds |
| Cost | Lower | Higher |
| Complexity | Simple | Complex (state, exactly-once) |
| Use case | Daily recommendations | Fraud detection, real-time pricing |
| Tools | Spark, Hadoop, dbt | Kafka, Flink, Spark Streaming |
| Feature freshness | Stale (hours old) | Fresh (<1 second old) |

**Lambda Architecture** (common in practice):
```
Data stream
    |
    +---> [Streaming layer] --> low latency, approximate features
    |
    +---> [Batch layer]     --> accurate, complete historical features
    |
    +---> [Serving layer]  --> merges both, serves queries
```

**Kappa Architecture** (simpler):
- Everything is a stream
- Replay historical data through the same streaming pipeline
- Simpler operationally, but requires stream system to handle replay


### Feature Store Design

A feature store is a central repository for ML features. It solves:
1. **Training-serving skew**: same feature computation code used everywhere
2. **Feature reuse**: team A's features available to team B
3. **Point-in-time correctness**: no data leakage from future
4. **Low-latency serving**: pre-computed features served in <10ms

```
Data Sources
    |
[Feature Engineering] (Spark/Flink)
    |
    +---> [Offline Store] (S3/BigQuery) -- for training
    |
    +---> [Online Store]  (Redis/DynamoDB) -- for serving
                |
             [Model] <-- features fetched at prediction time
```

**Feature store components:**
- **Registry**: metadata about each feature (owner, schema, freshness)
- **Transformation**: compute features from raw data
- **Storage**: offline (data lake) + online (key-value store)
- **Serving API**: `get_online_features(entity_ids, feature_names)`


### Model Serving Latency Optimization


In [ ]:
# Latency budget analysis for a typical ML serving request

def latency_budget_analysis():
    """Model the latency components of an ML serving call."""

    components = {
        'Network (client to LB)': 5,
        'Load balancer routing': 1,
        'Request parsing': 2,
        'Feature store lookup (online)': 8,
        'Feature assembly': 3,
        'Model inference (CPU)': 25,
        'Post-processing': 2,
        'Response serialization': 2,
        'Network (LB to client)': 5,
    }

    # Optimized with caching + GPU
    components_optimized = {
        'Network (client to LB)': 5,
        'Load balancer routing': 1,
        'Request parsing': 1,
        'Feature store lookup (cache hit)': 1,   # Redis L1 cache
        'Feature assembly': 1,
        'Model inference (GPU/quantized)': 5,    # GPU or TensorRT
        'Post-processing': 1,
        'Response serialization': 1,
        'Network (LB to client)': 5,
    }

    total = sum(components.values())
    total_opt = sum(components_optimized.values())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Original
    labels = list(components.keys())
    values = list(components.values())
    colors = ['#E74C3C' if v > 10 else '#F39C12' if v > 5 else '#27AE60'
              for v in values]
    bars = axes[0].barh(labels, values, color=colors)
    axes[0].set_xlabel('Latency (ms)')
    axes[0].set_title(f'Original Latency Budget\nTotal: {total}ms',
                      fontsize=12, fontweight='bold')
    for bar, val in zip(bars, values):
        axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                    f'{val}ms', va='center', fontsize=9)

    # Optimized
    labels_opt = list(components_optimized.keys())
    values_opt = list(components_optimized.values())
    colors_opt = ['#E74C3C' if v > 10 else '#F39C12' if v > 5 else '#27AE60'
                  for v in values_opt]
    bars_opt = axes[1].barh(labels_opt, values_opt, color=colors_opt)
    axes[1].set_xlabel('Latency (ms)')
    axes[1].set_title(f'Optimized Latency Budget\nTotal: {total_opt}ms',
                      fontsize=12, fontweight='bold')
    for bar, val in zip(bars_opt, values_opt):
        axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                    f'{val}ms', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('/tmp/latency_budget.png', dpi=100, bbox_inches='tight')
    plt.show()

    print(f"Original total latency: {total}ms")
    print(f"Optimized total latency: {total_opt}ms")
    print(f"Speedup: {total/total_opt:.1f}x")
    print()
    print("Key optimizations:")
    print("  1. Feature caching: 8ms -> 1ms  (cache hot user features in Redis)")
    print("  2. GPU inference:  25ms -> 5ms  (or CPU quantization via TensorRT/ONNX)")

latency_budget_analysis()


### Caching Strategies for ML

| Strategy | What to cache | TTL | Hit rate |
|---|---|---|---|
| User feature cache | Pre-computed user embeddings | 5 min | ~90% |
| Item feature cache | Item embeddings (change slowly) | 1 hour | ~99% |
| Prediction cache | Final recommendation list | 30 sec | ~70% |
| Query result cache | Top-K for popular queries | 10 min | ~60% |
| Model output cache | For deterministic models | 1 min | ~50% |

**Cache invalidation rules:**
- User features: invalidate on purchase, rating, or session end
- Item features: invalidate on price change, inventory, new reviews
- Predictions: invalidate on A/B test assignment change


### Trade-off Tables for ML Architecture Decisions


In [ ]:
# Generate a comprehensive trade-off comparison table

tradeoffs = {
    'Model Architecture': {
        'Logistic Regression': {'accuracy': 3, 'latency': 10, 'interpretability': 10,
                                'training_cost': 10, 'maintenance': 10},
        'Gradient Boosting': {'accuracy': 8, 'latency': 7, 'interpretability': 7,
                              'training_cost': 8, 'maintenance': 8},
        'Neural Network': {'accuracy': 10, 'latency': 5, 'interpretability': 3,
                           'training_cost': 4, 'maintenance': 5},
        'Transformer': {'accuracy': 10, 'latency': 3, 'interpretability': 2,
                        'training_cost': 2, 'maintenance': 4},
    },
    'Serving Infrastructure': {
        'Real-time API (CPU)': {'latency': 6, 'cost': 7, 'scalability': 6,
                                'freshness': 10, 'complexity': 7},
        'Real-time API (GPU)': {'latency': 9, 'cost': 4, 'scalability': 7,
                                'freshness': 10, 'complexity': 6},
        'Batch precompute': {'latency': 3, 'cost': 10, 'scalability': 10,
                             'freshness': 3, 'complexity': 8},
        'Streaming (Flink)': {'latency': 8, 'cost': 5, 'scalability': 8,
                              'freshness': 9, 'complexity': 3},
    }
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (category, options) in zip(axes, tradeoffs.items()):
    dims = list(list(options.values())[0].keys())
    x = np.arange(len(dims))
    width = 0.8 / len(options)
    colors_list = ['#2ECC71', '#3498DB', '#E74C3C', '#9B59B6', '#F39C12']

    for i, (name, scores) in enumerate(options.items()):
        vals = [scores[d] for d in dims]
        offset = (i - len(options)/2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=name,
               color=colors_list[i % len(colors_list)], alpha=0.85, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels([d.replace('_', ' ').title() for d in dims], rotation=20, ha='right')
    ax.set_ylabel('Score (1-10)')
    ax.set_ylim(0, 12)
    ax.set_title(f'{category} Trade-offs', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/tradeoffs.png', dpi=100, bbox_inches='tight')
plt.show()
print("Trade-off comparison chart generated.")


---
## Section 6 — 20 System Design Interview Questions with Frameworks

For each question, we use the 4-step framework:
**Problem --> Data --> Model --> Deployment**

These are real questions asked at Google, Meta, Apple, Netflix, Stripe, Airbnb, and Uber.


### Q1. Design a spam detection system for Gmail

**Step 1 — Problem**
- Business goal: reduce spam reaching inbox, minimize false positives (legit email in spam)
- Latency: classify within 500ms of email receipt
- Scale: 100B emails/day at Gmail scale
- Cost asymmetry: FP (legit email in spam) >> FN (spam in inbox) for most users
- Success metric: <0.1% FP rate, >99% recall on spam

**Step 2 — Data**
- Signals: email content, headers, sender reputation, IP reputation, user feedback (mark as spam)
- Labels: user actions (move to spam, mark not spam), Google's known spam list
- Volume: billions of labeled examples from user feedback
- Challenge: adversarial; spammers evolve to avoid detection

**Step 3 — Model**
- Two-stage: fast pre-filter (rules + logistic regression) then deep model for borderline
- Features: TF-IDF on content, sender domain reputation score, link analysis, header anomalies
- Model: LightGBM + text embeddings (fine-tuned BERT for content)
- Ensemble: combine rule-based and ML scores

**Step 4 — Deployment**
- Serving: synchronous, <500ms SLA, deployed as microservice
- Monitoring: FP/FN rates daily, feature drift, new spam patterns
- Retraining: weekly on recent data, hot-patching for new spam campaigns
- Feedback loop: user "mark as spam" / "not spam" signals retrain daily

---


### Q2. Design a recommendation system for Netflix

**Step 1 — Problem**
- Goal: maximize watch time and user satisfaction (long-term retention)
- Multiple surfaces: homepage, continue watching, similar titles, search results
- Latency: homepage load <200ms (precomputable), similar titles <50ms
- Scale: 200M users, 15k titles

**Step 2 — Data**
- Explicit: ratings (star ratings), thumbs up/down
- Implicit: watch time, completion rate, time to click, replays
- Context: device, time of day, subscription plan
- Cold start: new users get popularity-based + questionnaire
- Labels: watch >70% of title = positive; watched <10 min and stopped = negative

**Step 3 — Model**
- Two-stage: retrieval (collaborative filtering) + ranking (personalized LTR)
- Retrieval: matrix factorization or two-tower model, top 1000 candidates
- Ranking: XGBoost with 100s of features (user, item, context, interaction history)
- Row-level: different ranking for top row vs other rows
- Diversity: inject content from multiple categories to avoid filter bubbles

**Step 4 — Deployment**
- Precompute user embeddings nightly; refresh with recent activity in near-real-time
- A/B test framework: holdout groups for each experiment
- Guardrail metrics: total viewing hours, subscriber retention
- Personalization service: key-value lookup by user_id for pre-ranked items

---


### Q3. Design a real-time fraud detection system

**Step 1 — Problem**
- Detect fraudulent credit card transactions before authorization
- Latency: <100ms (must fit within payment processing window)
- Scale: 10k-100k TPS depending on company size
- Cost: false negative = fraud loss; false positive = customer friction
- Regulatory: must be explainable for disputes

**Step 2 — Data**
- Transaction: amount, merchant, time, location, card type
- Velocity features: spend in last 1h, 24h, 7d; # failed attempts
- Network/graph: merchant fraud rate, BIN (first 6 digits) fraud rate
- Device: device fingerprint, IP, geolocation
- Labels: chargebacks (30-90 day delay), real-time fraud reports
- Challenge: delayed labels, class imbalance (~1% fraud)

**Step 3 — Model**
- Lightweight model for <100ms: LightGBM or small NN (avoid transformers)
- Features must be precomputed: velocity counts stored in Redis
- Threshold tiers: auto-block / step-up auth / auto-approve
- Model explainability: SHAP values for dispute resolution

**Step 4 — Deployment**
- Feature pipeline: streaming (Flink/Kafka) for velocity features
- Serving: sync API, p99 latency <50ms
- Monitoring: PSI for score drift, daily fraud rate, chargeback rate
- Retraining: weekly with new labels; champion/challenger deployment
- Feedback: chargebacks trigger label update and weighted resampling

---


### Q4. Design the ML system for Uber Surge Pricing

**Step 1 — Problem**
- Predict demand and supply imbalance to set dynamic prices
- Goal: balance supply (driver availability) with demand (rider requests)
- Latency: prices update every few minutes; not pure real-time
- Scope: per hexagonal grid cell (H3 grid ~500m resolution)

**Step 2 — Data**
- Demand signals: historical ride requests by cell/time, events, weather
- Supply signals: driver GPS locations, driver acceptance rates
- External: events API, weather API, public holidays
- Labels: actual utilization rate, time-to-pickup (surrogate for supply/demand ratio)

**Step 3 — Model**
- Demand forecast: gradient boosting or LSTM per grid cell, 15-min horizon
- Supply forecast: simpler model (drivers respond to surge prices)
- Pricing logic: rule-based on (demand_forecast / supply_estimate) ratio
- Feedback loop: surge price affects both demand and supply — must model carefully

**Step 4 — Deployment**
- Batch-ish: recompute surge prices every 5 minutes per grid cell
- Edge cases: concerts, sports events (spike detection)
- Monitoring: driver utilization rate, rider wait time, cancellation rate
- Fairness: ensure surge doesn't disproportionately affect low-income areas
- A/B testing: hard due to interference between treatment/control cells (SUTVA violation)

---


### Q5. Design a content moderation system for Facebook/Instagram

**Step 1 — Problem**
- Detect and remove violating content: hate speech, violence, nudity, misinformation
- Latency: soft real-time (can take seconds; synchronous for new posts ok)
- Scale: 500M posts/day across text, images, video
- Stakes: false positives = free speech concerns; false negatives = harm

**Step 2 — Data**
- Multi-modal: text, image, video, audio, structured metadata
- Labels: policy violator decisions from human review team
- Active learning: route uncertain predictions to reviewers
- Languages: 100+ languages, unequal data quality

**Step 3 — Model**
- Multi-modal pipeline: separate models for text, image, video
- Text: fine-tuned multilingual BERT or LLaMA
- Image: CNN-based classifier (ResNet, ViT)
- Video: frame sampling + temporal aggregation
- Ensemble: combine scores, human review for borderline cases
- Zero-shot for new violation types (GPT-4 as policy expert)

**Step 4 — Deployment**
- Async post-upload pipeline, sync check for highest-risk users/content
- Tiered action: remove, reduce distribution, add warning label
- Appeals system: human review on request, model retraining from appeals
- Monitoring: removal rate by category, over-removal rate, appeals rate

---


### Q6. Design a Search Autocomplete system

**Step 1 — Problem**
- Predict the user's intended query from partial input
- Latency: <50ms for suggestions to appear as user types
- Scale: billions of queries/day; queries must feel personalized

**Step 2 — Data**
- Query logs: historical queries, click-through, query completion patterns
- User context: location, language, search history, device
- Labels: what query did user ultimately submit?

**Step 3 — Model**
- Candidate generation: prefix trie on top-N global queries
- Ranking: personalized re-ranking with user features
- Neural: character-level language model for rare queries
- Filtering: remove offensive completions

**Step 4 — Deployment**
- Trie loaded in RAM, updated nightly
- User-personalized features served from Redis
- Latency: trie lookup <5ms, ranking <20ms
- A/B test: measure completion rate and downstream search success

---


### Q7. Design an ETA (Estimated Time of Arrival) system for Uber/DoorDash

**Step 1 — Problem**
- Predict delivery or pickup time accurately
- Wrong ETA loses customer trust; underestimating hurts operations
- Multiple stages: pickup + transit + drop-off
- Real-time constraints: dynamic traffic, multiple active orders

**Step 2 — Data**
- Route: distance, road segments, historical speeds by time-of-day
- Real-time: current traffic (GPS pings from drivers), events
- Order: restaurant prep time distribution, order complexity
- Historical: actual vs predicted ETA, cancellation data

**Step 3 — Model**
- Stage 1 route: graph neural network on road network OR gradient boost on features
- Stage 2 restaurant: survival model (time-to-event) for prep time
- Stage 3 aggregate: combine stages with uncertainty intervals (quantile regression)
- Uncertainty: show "25-35 min" not just "30 min" when uncertain

**Step 4 — Deployment**
- Continuous learning: update road speed estimates from live GPS pings
- Monitoring: MAPE by time-of-day, area, order type
- Business rules: add buffer during peak hours, events

---


### Q8. Design a loan default prediction system

**Step 1 — Problem**
- Predict probability of loan default within 12 months
- Latency: decision within minutes (can be near-real-time, not hard real-time)
- Regulatory: must be fair (ECOA, Fair Housing Act), explainable (adverse action reasons)
- Asymmetric costs: false negative (approve bad loan) = large loss; FP = lost revenue

**Step 2 — Data**
- Credit bureau: FICO score, payment history, utilization, derogatory marks
- Application: income, employment, loan amount, purpose
- Internal: existing customer history (if applicable)
- Labels: default within 12 months (observe over time)
- Challenge: survivorship bias (only approved loans have outcomes)

**Step 3 — Model**
- Gradient boosting (LightGBM) with regularization
- Fairness constraints: equal opportunity constraint across protected groups
- Explainability: SHAP for adverse action codes (required by law)
- Calibration: probabilities must be well-calibrated for risk-based pricing

**Step 4 — Deployment**
- Sync API with human review queue for borderline cases
- Fair lending monitoring: disparate impact analysis quarterly
- Model validation: separate team validates model before deployment
- Documentation: model risk management, SR 11-7 compliance

---


### Q9. Design a news feed ranking system for Twitter/X

**Step 1 — Problem**
- Rank tweets to maximize engagement (but also long-term user health)
- Tension: engagement optimization can amplify outrage/misinformation
- Latency: feed must load in <300ms (precomputable)
- Scale: 500M users, 500M tweets/day

**Step 2 — Data**
- Engagement signals: likes, retweets, replies, shares, clicks, dwell time
- Social graph: who you follow, who they follow (2-hop)
- Content: text embeddings, media type, entities mentioned
- Freshness: tweet age strongly affects relevance
- Labels: click, like, retweet (multi-task learning targets)

**Step 3 — Model**
- Candidate generation: "For You" (interest-based), "Following" (chronological subset)
- Ranking: multi-task neural network predicting probability of each engagement type
- Weighted objective: combine engagement metrics with health metrics
- Diversity: down-rank if too many tweets from same author

**Step 4 — Deployment**
- Precompute user embeddings; real-time tweet embeddings
- Multi-armed bandit for exploration (new content discovery)
- Monitoring: engagement rates, report rates, account health scores
- Transparency: public model card describing ranking factors

---


### Q10. Design an ad click-through rate (CTR) prediction system

**Step 1 — Problem**
- Predict P(click | user, ad, context) for ad auction
- Latency: <10ms (tight SLA for ad auctions)
- Scale: millions of ads, hundreds of billions of predictions/day
- Revenue directly proportional to CTR accuracy

**Step 2 — Data**
- User: demographics, interest categories, historical click behavior
- Ad: creative features, advertiser, product category, bid
- Context: page content, time, device, placement
- Labels: click/no-click (100ms cookie window), conversion (delayed)
- Volume: petabytes of log data, heavily imbalanced (CTR ~0.1-5%)

**Step 3 — Model**
- FTRL (Follow the Regularized Leader) for online learning (Google's approach)
- Deep & Wide: wide component (memorization) + deep component (generalization)
- Feature interactions: product features with user features (DeepFM, DCN)
- Calibration: model must output calibrated probabilities for auction math

**Step 4 — Deployment**
- Online learning: model updated with every click/impression in real-time
- Serving: highly optimized C++ inference, model quantized to INT8
- A/B testing: hold-out by user ID to measure true lift
- Monitoring: calibration, logloss, business RPM (revenue per thousand)

---


### Q11-Q15: More Questions (Condensed Format)

**Q11. Design a customer lifetime value (LTV) prediction model**
- Problem: regression on revenue over next 12 months; used for acquisition bidding
- Data: purchase history, product affinity, discount sensitivity, support tickets
- Model: quantile regression (predict 50th and 90th percentile, not just mean)
- Deploy: batch, retrain monthly, monitor coverage and calibration by segment

**Q12. Design a churn prediction system**
- Problem: predict which users will cancel subscription in next 30 days
- Data: usage patterns, support interactions, billing events, NPS scores
- Model: gradient boosting on recency/frequency/monetary features
- Deploy: daily batch scores, trigger retention campaigns via CRM integration

**Q13. Design an image similarity search system for Pinterest**
- Problem: given an image, find visually similar images in 1B+ image corpus
- Data: user pins, image metadata, engagement signals
- Model: CNN encoder (ResNet/ViT) for image embeddings, FAISS for ANN search
- Deploy: offline embed all images, online: embed query image + ANN lookup

**Q14. Design a machine translation system**
- Problem: translate text between 100+ language pairs
- Data: parallel corpora, web-crawled text, user corrections
- Model: Transformer (encoder-decoder), fine-tuned per language pair or multilingual
- Deploy: beam search decoding, latency-quality trade-off via beam width

**Q15. Design a voice assistant intent classification system**
- Problem: classify user speech into intent (play music, set timer, search web)
- Data: transcribed speech + labels, dialog history
- Model: ASR (Whisper) -> text classification (BERT fine-tune) -> slot filling
- Deploy: on-device ASR + cloud classification for complex intents

---


### Q16-Q20: Infrastructure and Advanced Questions

**Q16. How would you detect data drift in production?**
Key answer: Use Population Stability Index (PSI) for feature distributions,
Kolmogorov-Smirnov test for numerical features, chi-squared test for categorical.
Monitor model output distribution separately from input features. Set up automated
alerts and retraining triggers when PSI > 0.25 or prediction distribution shifts >15%.

**Q17. How do you handle the cold start problem at scale?**
Key answer: (1) New user: progressive profiling (ask 3-5 preference questions at onboarding),
fallback to demographic-based or regional popular items; (2) New item: use content features
(CB filtering), bootstrap with weighted popular items, use exploration strategy;
(3) Long-term: convert cold items to warm using cascade of models.

**Q18. How do you ensure your ML model is fair?**
Key answer: (1) Define fairness metric upfront (demographic parity, equal opportunity,
individual fairness — they conflict, choose based on use case); (2) Slice analysis across
protected groups during development; (3) Adversarial debiasing or constrained optimization
during training; (4) Ongoing monitoring with statistical tests; (5) Human audit on
high-stakes decisions.

**Q19. How would you debug a model that performs well offline but poorly online?**
Key answer: (1) Training-serving skew check — verify features are identical;
(2) Label leakage check — any future data in training features?; (3) Sample selection
bias — is training distribution representative of production traffic?;
(4) Position bias (for ranking) — did you correct for which items were shown?;
(5) Feedback loop — are model predictions affecting training data?

**Q20. How do you decide when to retrain vs. rebuild a model?**
Key answer: Retrain when performance degrades due to distribution shift but the task/data
schema is stable. Rebuild when: (1) business requirements change fundamentally;
(2) better architecture exists (e.g., upgrade from GBM to transformer);
(3) training data schema changes significantly; (4) model performance improvement
from new architecture > cost of full retraining pipeline. Retrain on a schedule
(weekly/monthly) plus triggered by monitoring alerts.

---


---
## Summary — ML System Design Cheat Sheet

### The Framework (memorize this)
```
1. PROBLEM FRAMING
   - Business objective and success metrics
   - Latency / scale / fairness constraints
   - Online vs offline metrics

2. DATA STRATEGY
   - Sources, labeling, pipelines
   - Split strategy (time-based for temporal data)
   - Class imbalance, distribution analysis

3. MODEL SELECTION
   - Start simple (LR / GBM), add complexity only if needed
   - Feature engineering > model complexity in most cases
   - Evaluation: multiple metrics, slice analysis

4. DEPLOYMENT & OPERATIONS
   - Serving architecture (batch/streaming/real-time)
   - Monitoring (data drift, prediction drift, business metrics)
   - Feedback loops and retraining strategy
```

### Red Flags to Avoid in Interviews
- Jumping to model before understanding the problem
- Only mentioning neural networks (show breadth)
- Ignoring data quality and labeling
- Forgetting latency and scale constraints
- Not discussing failure modes and monitoring
- Proposing a solution without trade-off analysis

### Interview Scoring Rubric
| Dimension | What they look for |
|---|---|
| Problem clarity | Did you ask good clarifying questions? |
| Technical depth | Do you know multiple approaches and their trade-offs? |
| Scale awareness | Did you consider latency, throughput, cost? |
| Production mindset | Did you address monitoring, drift, and failure modes? |
| Communication | Did you structure the answer clearly? |

### Key Papers to Reference
- "Practical Lessons from Predicting Clicks on Ads at Facebook" (He et al., 2014)
- "Deep Neural Networks for YouTube Recommendations" (Covington et al., 2016)
- "Real-time Machine Learning: The Missing Pieces" (Klaise et al.)
- "Hidden Technical Debt in Machine Learning Systems" (Sculley et al., 2015)
- "Challenges in Deploying Machine Learning" (Paleyes et al., 2022)


## Final Notes

This notebook was designed as a comprehensive reference for senior ML engineer,
ML researcher, and applied scientist interviews at top tech companies.

**What to study next:**
- Implement a full two-tower retrieval model with PyTorch
- Build a streaming feature pipeline with Kafka + Flink
- Study FAISS for billion-scale approximate nearest neighbor search
- Read Chip Huyen's "Designing Machine Learning Systems" (O'Reilly)
- Practice on ml-interview-prep resources and past interview writeups

**Time allocation in a 45-minute interview:**
- 0-5 min: clarify requirements
- 5-15 min: data strategy
- 15-30 min: model selection and feature engineering
- 30-40 min: deployment, monitoring, scale
- 40-45 min: wrap up, answer questions, discuss trade-offs

Good luck!
